# PoetryDuel-GPT v2.1 - Single-Stage Doi Tho Training

**One corpus, one stage. Train on couplet-to-couplet dueling only.**
| Step | Time |
| Download data + train tokenizer | ~3 min |
| Single-stage training (10K steps) | ~3 hr T4 |
| Generate poetry | ~1 min |
Data is in data.zip on your Drive. No preprocessing needed.


In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm datasets
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print('\n⚠️  Run cells in order. Each 🔍 cell verifies the previous step.')


In [ ]:
# @title 2. Download Data + Train Tokenizer (~3 min)
from google.colab import drive
drive.mount('/content/drive')

# Copy data.zip from Drive
!cp /content/drive/MyDrive/poetry-dual-gpt/data.zip /content/poetry-dual-gpt/ 2>/dev/null
!unzip -o data.zip
!ls data/

# Train BPE tokenizer on doi_tho corpus
%cd /content/poetry-dual-gpt
!python src/train_bpe.py --corpus data/doi_tho_corpus.txt

# Verify
from tokenizers import Tokenizer
tok = Tokenizer.from_file('tokenizer/poetry_bpe.model')
print(f'Vocab: {tok.get_vocab_size():,}')
for name in ['<|pad|>', '<|start|>', '<|reply|>', '<|end|>', '[DOI_THO]', '<|linebreak|>']:
    n = len(tok.encode(name).ids)
    print(f'  {name:20s} subwords={n} {"OK" if n==1 else "FAIL"}')

print("\nReady for training.")


In [ ]:
# @title 3. Train — Single Stage on Doi Tho (~3 hr T4)
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU"

%cd /content/poetry-dual-gpt
import glob

# One corpus: doi_tho_corpus.txt (998K couplet-to-couplet pairs)
CORPUS = 'data/doi_tho_corpus.txt'

latest = sorted(glob.glob("checkpoints/doi_tho_step_*.pt"))
if latest:
    print(f"Resuming from {latest[-1]}")
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS} --resume {latest[-1]}
else:
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS}

print("\nTraining complete -> checkpoints/doi_tho_best.pt")


In [ ]:
# @title Verify: Test Generation
%cd /content/poetry-dual-gpt
import os

ckpt = 'checkpoints/doi_tho_best.pt'
if not os.path.exists(ckpt):
    # try latest step checkpoint
    import glob
    candidates = sorted(glob.glob('checkpoints/doi_tho_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 1000000:
            ckpt = c
            break

if not os.path.exists(ckpt):
    print('No checkpoint found. Run training first.')
else:
    print(f'Testing: {ckpt}')
    print('\n--- Doi tho (pipe-separated couplet) ---')
    !python src/sample.py --checkpoint {ckpt} \
        --prompt "Than em nhu chen lua dong | Phat pho duoi ngon nang hong ban mai" \
        --num_samples 3 --device cuda
    
    print('\nExpected: 2-line Vietnamese couplet with 6+8 syllables each.')


In [ ]:
# @title Generate Poetry (custom prompt)
%cd /content/poetry-dual-gpt

# Change the prompt to your own couplet!
!python src/sample.py \
    --checkpoint checkpoints/doi_tho_best.pt \
    --prompt "Than em nhu chen lua dong | Phat pho duoi ngon nang hong ban mai" \
    --temperature 0.75 \
    --top_p 0.92 \
    --num_samples 3 \
    --device cuda


In [ ]:
# @title Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/poetry-dual-gpt'
!mkdir -p {DRIVE_DIR}/checkpoints

# Save tokenizer + checkpoints
!cp -v tokenizer/poetry_bpe.model {DRIVE_DIR}/
import glob
for ckpt in sorted(glob.glob('checkpoints/doi_tho_*.pt')):
    !cp -v {ckpt} {DRIVE_DIR}/checkpoints/
print('All saved to Drive -> poetry-dual-gpt/')
